# SNA Group 14 — **CHỌN GIAI ĐOẠN E1–E4** (Colab v2.1)

**Bước trước:** Notebook `01_START_HERE_Colab.ipynb` phải tạo xong Gemini item/user profiles, embeddings và `package` PASS.

Chọn **một giai đoạn** và các tham số trong ô bên dưới. Notebook này **chỉ tạo file Colab mới trên Drive**, tuyệt đối không tự động chạy GPU. Đóng notebook điều khiển sau khi chuẩn bị và mở notebook đã tạo bằng Google Colab → Runtime → Run all.

Làm việc **theo thứ tự**: E1 train → E1 export → Most Popular → E1 summary → E2 → E3A → E3B → E3C train/export/analysis → E3D → E4 train/export/analysis. Chọn đúng seed/model/layer. E3C reuse **E1 mới**, E4 recovery phải thuộc nhánh mới. Các file GPT-3.5 cũ vẫn nguyên vẹn.

In [ ]:
# ĐIỀN THÔNG TIN TƯƠNG ỨNG VỚI 01_START_HERE
ZIP_ON_DRIVE='/content/drive/MyDrive/SNA_Group14_Amazon_Gemini_Colab_v2_1.zip'
ORIGINAL_PROJECT_ROOT='/content/drive/MyDrive/Data_SNA_Group14'
RUN_ID='gemini_colab_pilot_001'
# Mỗi lần chọn 1 stage và model/seed/layer cần chạy (khi có).
STAGE='e1-train'          # e1-train, e1-export, e1-popular, e1-summary, e2, e3a,
                          # e3b, e3c-train, e3c-export, e3c-analysis, e3d, e4-train,
                          # e4-export, e4-analysis
MODEL='lightgcn'         # e1: 'lightgcn'|'lightgcn_plus'; E4: 'LightGCN'|'RLMRec-Con'
SEED=2023                # e1/e3a: 2023,2024,2025
MODE='user_only'          # e3a: 'user_only'|'item_only'
LAYER=1                 # E4: 1,2,3

## 1. Mount Drive, mở pipeline và xác minh repo

In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, zipfile

drive.mount('/content/drive')
BRANCH=Path(ORIGINAL_PROJECT_ROOT)/'experiments_llm_colab_v21'/RUN_ID
CFG=BRANCH/'settings/colab_config.json'
assert CFG.is_file(), f'Không thấy cấu hình nhánh {RUN_ID}. Hãy chạy START_HERE cho đến bước tạo config: {CFG}'
ZIP=Path(ZIP_ON_DRIVE)
if not ZIP.is_file():
    print('ZIP không thấy trên Drive: upload ZIP vào MyDrive hoặc vào /content của runtime')
    from google.colab import files
    uploaded=files.upload()
    assert len(uploaded)==1 and next(iter(uploaded)).endswith('.zip')
    ZIP=Path('/content')/next(iter(uploaded))
TMP=Path('/content/sna_group14_colab_v21');TMP.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    for m in z.infolist():
        target=(TMP/m.filename).resolve()
        assert target==TMP.resolve() or target.is_relative_to(TMP.resolve()),'Unsafe ZIP entry'
    z.extractall(TMP)
PIPELINE=TMP/'SNA_Group14_Amazon_Gemini_Colab_v2_1'
assert (PIPELINE/'run.py').is_file(), 'Sai ZIP pipeline'
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(PIPELINE/'requirements.txt')],check=True)

def git_checkout(url,dest,sha):
    repo=Path(dest)
    if not repo.exists():
        subprocess.run(['git','clone',url,str(repo)],check=True)
        subprocess.run(['git','-C',str(repo),'checkout','--detach',sha],check=True)
    assert (repo/'.git').exists(),'Git repository missing: '+str(repo)
    actual=subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
    assert actual==sha, f'Wrong Git SHA: {repo} {actual}'
git_checkout('https://github.com/HKUDS/RLMRec.git','/content/RLMRec','22413752246de3dee8ab0d509f7f7a8889080f95')
git_checkout('https://github.com/trungkien1502/project_sna_group14.git','/content/project_sna_group14','c17caeb4448a8add7939510411339af918d258bb')
print('Source repos pinned. BRANCH:',BRANCH)

## 2. Xem kế hoạch chạy và trạng thái có sẵn (không huấn luyện)

In [ ]:
subprocess.run([sys.executable,str(PIPELINE/'run.py'),'--config',str(CFG),'run-all'],cwd=PIPELINE,check=True)

## 3. Tạo notebook cho một stage đã chọn (không tự thực thi)

In [ ]:
# Trình điều khiển CHỈ TẠO notebook, KHÔNG train/đánh giá tự động.
args=['run-stage','--stage',STAGE]
if STAGE in ('e1-train','e1-export'):
    assert MODEL in ('lightgcn','lightgcn_plus')
    assert SEED in (2023,2024,2025)
    args+=['--model',MODEL,'--seed',str(SEED)]
elif STAGE=='e3a':
    assert MODE in ('user_only','item_only') and SEED in (2023,2024,2025)
    args+=['--mode',MODE,'--seed',str(SEED)]
elif STAGE=='e4-train':
    assert MODEL in ('LightGCN','RLMRec-Con') and LAYER in (1,2,3)
    args+=['--model',MODEL,'--layer',str(LAYER)]
cmd=[sys.executable,str(PIPELINE/'run.py'),'--config',str(CFG),*args]
print('PREPARING:',STAGE,MODEL,SEED,MODE,LAYER)
subprocess.run(cmd,cwd=PIPELINE,check=True)
name=STAGE+('-'+str(MODEL) if STAGE in ('e1-train','e1-export','e4-train') else '')+('-'+str(MODE) if STAGE=='e3a' else '')+('-'+str(LAYER) if STAGE=='e4-train' else '')+('-'+str(SEED) if STAGE in ('e1-train','e1-export','e3a') else '')
notebook=BRANCH/'runtime/notebooks'/f'{name}.ipynb'
assert notebook.is_file(),notebook
print('\nNOTEBOOK READY:',notebook)
print('Google Drive → tìm file .ipynb ở đường dẫn này → Mở bằng Google Colab → Runtime → Run all')
print('Chạy stage xong, kiểm tra output PASS trong nhánh Drive mới rồi mới sang stage tiếp theo.')